# 13 — The complete decision assistant
This is the same Crew and Flow from earlier lessons. The CLI supplies one
request, a customer identity, and an explicit approval flag for a local mock
email. No external email service is connected.



**Basic lesson 13 of 13.**


## Problem

The pieces must work together behind one entry point.


## Concept

The CLI invokes the existing CrewAI Flow; it does not duplicate decision logic.

The interface passes a request into one application entry point. Memory enriches context, the Flow owns control, the Crew researches, and the action gate controls the only side effect.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out   ◀ YOU ARE HERE
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
CLI request ──▶ run_assistant ──▶ scoped Memory
                    │                  │
                    ▼                  ▼
                DecisionFlow ──▶ Research Crew ──▶ DecisionReport
                    │
                    └──▶ approval gate ──▶ local mock action
```

question/customer/approval → Flow → Crew → report/action route. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Parse the CLI request and reject an approval flag without an action request

Parse the CLI request and reject an approval flag without an action request.


In [ ]:
import argparse
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.app import run_assistant

parser = argparse.ArgumentParser()
parser.add_argument("--question", default="Should Northstar AI pilot Meridian AI?")
parser.add_argument("--customer", default="customer-a")
parser.add_argument("--remember", help="Store a customer constraint before this run")
parser.add_argument("--request-email", action="store_true")
parser.add_argument("--approve-email", action="store_true")
args, _ = parser.parse_known_args()
if args.approve_email and not args.request_email:
    parser.error("--approve-email requires --request-email")



## Run the end-to-end Flow
Inspect the recommendation, the citation IDs, and the selected route.
Try --request-email first, then explicitly add --approve-email.



## Step 2 — Run the same Flow used earlier

Run the same Flow used earlier; inspect report and action route separately.


In [ ]:
flow, output = run_assistant(
    question=args.question,
    customer_id=args.customer,
    remember=args.remember,
    request_email=args.request_email,
    approved=args.approve_email,
)
print(output)
print("Report:", flow.state.report)
print("Action:", flow.state.action_result or "none")



## Bonus scenarios — same assistant, different questions
The Flow does not care what the question is, as long as the documents can
answer it. Re-run the cell above with one of these using `--question`:
- "Does Meridian AI support private VPC deployment?"
- "What is the biggest security risk of piloting Meridian AI?"
- "Is the $45,000 annual benefit realistic given the pricing document?"
Then compare the evidence IDs the report cites for each one.


## Failure mode to know about

A good final report does not imply the action route is safe. Exercise the no-approval path, approved path, and replay of the same action ID separately.


## Production implication

The CLI is the workshop's minimal interface; an API or UI can call the same entry point later. Do not reimplement the Flow's approval logic in that interface.


## Your turn

Run a report, then request a mock email without and with approval.

**Checkpoint:** The report has sources and only explicit approval records the mock action.
